In [5]:
import pandas as pd

df = pd.read_csv('halloween_vendas.csv')

df.head(3)


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110.0,57.00,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


In [6]:
print("Total de linhas e colunas:", df.shape)


Total de linhas e colunas: (1500, 14)


In [ ]:
import re

def limpar_categoria(texto):
    if pd.isna(texto) or str(texto).strip() == '':
        return 'sem categoria'
    
    limpo = str(texto).strip().lower()
    
    limpo = re.sub(r'[^a-záéíóúâêîôûãõç\s]', '', limpo).strip()
    
    if limpo == '':
        return 'sem categoria'
        
    return limpo


In [10]:

df['categoria'] = df['categoria'].apply(limpar_categoria)

df.head(3)


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,fantasia,Loja física,Maringá,3,110.0,57.00,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


In [ ]:
def padronizar_canal(canal):
    c = str(canal).strip().lower()
    if 'mercado' in c:
        return 'Mercado'
    elif 'loja' in c or 'física' in c or 'fisica' in c:
        return 'Loja física'
    elif 'web' in c:
        return 'Web'
    return canal

df['canal_venda'] = df['canal_venda'].apply(padronizar_canal)

print("Canais de venda padronizados com sucesso!")


Canais de venda padronizados com sucesso!


In [11]:
print("--- DISTRIBUIÇÃO DAS CATEGORIAS ---")
print(df['categoria'].value_counts())

print("\n--- DISTRIBUIÇÃO DOS CANAIS DE VENDA ---")
print(df['canal_venda'].value_counts())


--- DISTRIBUIÇÃO DAS CATEGORIAS ---
categoria
fantasia         979
acessorio        503
sem categoria     18
Name: count, dtype: int64

--- DISTRIBUIÇÃO DOS CANAIS DE VENDA ---
canal_venda
Web            666
Loja física    551
Mercado        283
Name: count, dtype: int64


In [12]:
df['quantidade_numerica'] = pd.to_numeric(df['quantidade'], errors='coerce')
df['preco_numerico'] = pd.to_numeric(df['preco_unitario'], errors='coerce')

filtro_qtd_invalida = (df['quantidade_numerica'] <= 0) | (df['quantidade_numerica'].isna())

filtro_preco_invalido = (df['preco_numerico'] < 0) | (df['preco_numerico'].isna())

print(f"Total de pedidos com quantidade inválida: {filtro_qtd_invalida.sum()}")
print(f"Total de pedidos com preço inválido: {filtro_preco_invalido.sum()}")

pedidos_afetados = df[filtro_qtd_invalida | filtro_preco_invalido]
print(f"Total de pedidos únicos afetados: {len(pedidos_afetados)}")


Total de pedidos com quantidade inválida: 7
Total de pedidos com preço inválido: 12
Total de pedidos únicos afetados: 19


In [14]:
print("--- EXEMPLOS DE PEDIDOS COM ERRO NUMÉRICO ---")
colunas_exibir = ['pedido_id', 'quantidade', 'quantidade_numerica', 'preco_unitario', 'preco_numerico']
print(pedidos_afetados[colunas_exibir].head(5))

df_limpo = df[~(filtro_qtd_invalida | filtro_preco_invalido)].copy()

print(f"\nTotal de linhas originais: {len(df)}")
print(f"Total de linhas limpas e válidas: {len(df_limpo)}")
print(f"Linhas removidas: {len(df) - len(df_limpo)} (1.27% da base)")


--- EXEMPLOS DE PEDIDOS COM ERRO NUMÉRICO ---
     pedido_id  quantidade  quantidade_numerica  preco_unitario  \
23        1500           3                    3             NaN   
164        630          -1                   -1            22.0   
199        625           2                    2             NaN   
216       1000           1                    1             NaN   
239       1250           1                    1             NaN   

     preco_numerico  
23              NaN  
164            22.0  
199             NaN  
216             NaN  
239             NaN  

Total de linhas originais: 1500
Total de linhas limpas e válidas: 1481
Linhas removidas: 19 (1.27% da base)


In [15]:
sem_entrega = df_limpo[df_limpo['data_entrega'].isna() | (df_limpo['data_entrega'] == '')]

print("--- INVESTIGAÇÃO DA HIPÓTESE DA GESTÃO ---")
print(f"Total de pedidos sem data de entrega: {len(sem_entrega)}")

print("\nStatus real desses pedidos sem data de entrega:")
status_contagem = sem_entrega['status_pedido'].value_counts()
print(status_contagem)

contraexemplos = sem_entrega[sem_entrega['status_pedido'] == 'Concluído']
print(f"\nTotal de contraexemplos (Concluídos mas sem data de entrega): {len(contraexemplos)}")
print("Exemplos de IDs desses pedidos:")
print(contraexemplos['pedido_id'].head(5).tolist())

if len(contraexemplos) > 0:
    print("\n👉 CONCLUSÃO: A hipótese da gestão é FALSA! Existem pedidos entregues/concluídos sem registro no sistema.")
else:
    print("\n👉 CONCLUSÃO: A hipótese da gestão é VERDADEIRA.")


--- INVESTIGAÇÃO DA HIPÓTESE DA GESTÃO ---
Total de pedidos sem data de entrega: 139

Status real desses pedidos sem data de entrega:
status_pedido
Cancelado    108
Concluído     31
Name: count, dtype: int64

Total de contraexemplos (Concluídos mas sem data de entrega): 31
Exemplos de IDs desses pedidos:
[1170, 990, 740, 45, 720]

👉 CONCLUSÃO: A hipótese da gestão é FALSA! Existem pedidos entregues/concluídos sem registro no sistema.


In [16]:
from datetime import datetime

def formatar_data_br(valor):
    try:
        dt = pd.to_datetime(valor)
       
        return dt.strftime('%d/%m/%Y')
    except:
        
        return 'Data Inválida'

df_limpo['data_venda_br'] = df_limpo['data_venda'].apply(formatar_data_br)

datas_invalidas = (df_limpo['data_venda_br'] == 'Data Inválida').sum()
print(f"Total de datas de venda corrompidas: {datas_invalidas}")

df_datas_validas = df_limpo[df_limpo['data_venda_br'] != 'Data Inválida'].copy()
df_datas_validas['ano_mes'] = pd.to_datetime(df_datas_validas['data_venda']).dt.to_period('M')

print("\n--- PICOS DE VENDA POR MÊS (SAZONALIDADE DO HALLOWEEN) ---")
print(df_datas_validas['ano_mes'].value_counts().sort_index())


Total de datas de venda corrompidas: 4

--- PICOS DE VENDA POR MÊS (SAZONALIDADE DO HALLOWEEN) ---
ano_mes
2024-08     75
2024-09    104
2024-10    388
2024-11    172
2025-08     62
2025-09     92
2025-10    405
2025-11    179
Freq: M, Name: count, dtype: int64
